
# 디지털정보격차 실태조사 전처리 문서

> **데이터**: 한국지능정보사회진흥원 디지털정보격차 실태조사 (2024~2025년)  
> **대상 그룹**: 일반국민, 농어민, 장애인, 저소득층, 고령층 (5개 그룹), 북한이탈주민과 결혼이민자는 제외  
> **목적**: 디지털 역량 수준 세분화 모델 학습을 위한 전처리

---

## 1. 전처리 전체 흐름

```
원천 데이터 로드 (읽기 전용)
        │
        ▼
Step 1  공백(' ') → NA 변환
        │
        ▼
Step 2  불필요한 컬럼 제거
        │  ET 컬럼, 응답ID, TYP, 9997 포함 컬럼
        │  100% 결측 컬럼, Q4A 계열, ADQ3A, ADQ8_1T
        │
        ▼
Step 3  특수코드 처리
        │  일반 컬럼: 9998/9999 값 → NA
        │  flag 컬럼: 9998/9999 포함 컬럼 결측 → 0
        │
        ▼
Step 4  구조적 결측 → 0
        │  Q12~Q19 (인터넷 비이용자 해당없음)
        │
        ▼
Feature Engineering
        │  Q5K1, Q4B, ADQ8A, Q27, Q28, Q31, Q2K1, Q2K2
        │
        ▼
공통 변수 매핑
        │  그룹별 컬럼명 → 연령, 성별, 직업, 학력, 가구구성형태, 가구소득, 거주지역
        │  GROUP, YEAR 컬럼 추가
        │
        ▼
train(70%) / val(15%) / test(15%) 분할
        │  stratify=GROUP 으로 그룹 비율 유지
        │
        ▼
수치형 변환 + Q4C, Q11 결측 중앙값 대체
        │  train 기준 중앙값 계산 → 세 세트 동일 적용
        │
        ▼
train.csv / val.csv / test.csv 저장
```

---

## 2. Step별 전처리 내역

### Step 1. 공백/빈문자열 → NA 변환

- **대상**: 전체 컬럼
- **이유**: 엑셀에서 응답 없는 셀이 진짜 빈 셀(NaN)이 아니라 공백 문자(' ')로 저장돼 있음. 그대로 두면 pandas가 결측을 0개로 인식해서 이후 모든 결측 처리가 무의미해짐
- **처리**: `df.replace(' ', pd.NA)`

---

### Step 2. 불필요한 컬럼 제거

아래 표 참고

---

### Step 3. 특수코드 처리

- **9998/9999 값 → NA**: 일반 수치형 컬럼에서 9998(모름/무응답), 9999(비해당)은 실제 응답값이 아님. 그대로 두면 모델이 매우 큰 숫자로 학습함
- **flag 컬럼 결측 → 0**: 컬럼명에 9998/9999가 포함된 컬럼(`Q27C_9998` 등)은 "해당 항목을 선택했는가"를 나타내는 flag 컬럼. NaN = 해당없음 = 0

---

### Step 4. 구조적 결측 → 0

- **대상**: `Q12~Q19` 계열 (검색/이메일/SNS/생활서비스/경제활동/사회참여/심화기술 이용 비율)
- **이유**: 인터넷 비이용자는 아예 질문을 안 받아서 NaN이 됨. 이 NaN의 의미가 "이용 안 함 = 0"이 확실함

---

### Step 5. 수치형 변환 + 결측 중앙값 대체 (분할 후)

- **수치형 변환**: 공백→NA 처리 후 object 타입으로 남은 컬럼을 `pd.to_numeric`으로 변환
- **Q4C, Q11 결측 중앙값 대체**: train set 기준으로만 중앙값 계산 후 val/test에 동일 적용 (데이터 누수 방지)

---

## 3. 제거한 컬럼 목록
 **파생변수 생성: AI인지, AI사용빈도, AI도움정도**
| 컬럼 | 이유 |
|---|---|
| `_ET` 로 끝나는 컬럼 | 기타 직접입력 주관식 컬럼. 100% 공백으로 정보 없음 |
| `응답ID` | 단순 식별자. 모델 학습에 쓰면 안 됨 |
| `TYP` | 그룹 구분 코드. GROUP 컬럼으로 대체 |
| `9997` 포함 컬럼 | 기타 선택 여부/텍스트. 선택자 극소수(1~43명)로 정보량 없음 |
| 100% 결측 컬럼 | 값이 하나도 없는 컬럼. 정보 없음 |
| `Q4A_*` 계열 | 인터넷/모바일 요금 금액. 모름 응답자가 많고, 요금 금액보다 Q4C(요금 부담 정도)가 디지털 역량과 더 관련있음. 모름 flag 컬럼(9999)도 함께 제거 |
| `ADQ3A` | 직업 고용주/피고용주 구분. 결측 72.9%, ADQ3(직업)으로 이미 커버됨 |
| `ADQ8_1T` | 가구구성형태 기타 텍스트. 결측 14.3%, 기타 응답이라 정보량 없음 |
| `ADQ8A5~8` | 다인가구 구성원 유형. 선택자 3~35명으로 극소수 |
| `Q2K1_*` 계열 | Q2K1_1_1은 98%가 1로 상수에 가까워 정보량 없음. Q2K1_1_2/3은 선택자 극소수 |
| `Q27A_*`, `Q27B_*`, `Q27C_*` 원본 | 합산 후 AI_인지, AI_사용빈도, AI_도움정도 컬럼으로 대체(대체 후 기존 열 삭제) |
| `BDQ_*`, `CDQ_*`, `DDQ_*` 원본 | 공통 변수(연령, 성별 등) 매핑 후 join='inner'로 제거 |

---

## 4. Feature Engineering

| 대상 | 원본 정보 | 처리 방식 | 이유 |
|---|---|---|---|
| `Q5K1_*` | 혼자 해결하는 방법 (포털/유튜브/앱 등) | 이진변환 (값있음=1, NaN=0) | 어떤 방법을 쓰는지가 중요하므로 합산 안 함 |
| `Q4B_1_1`, `Q4B_2_1` | 결합상품 가입 여부 (인터넷/모바일) | 이진변환 (값있음=1, NaN=0) | 가입=1, 미가입=0 |
| `ADQ8A1~4` | 다인가구 구성원 유형 | 이진변환 (값있음=1, NaN=0) | 유형별 선택 여부가 중요 |
| `Q27A_*` (8개 카테고리) | AI 서비스 인지 여부 | 이진변환 후 합산 → `AI_인지` (0~8) | 8개 카테고리 중 몇 개를 인지하는지. 높을수록 AI 인지도 높음 |
| `Q27B_*` (8개 카테고리) | AI 서비스 사용 빈도 | NaN=0 후 합산 → `AI_사용빈도` (0~32) | 미인지=미사용이므로 NaN을 0으로. 높을수록 AI 자주 사용 |
| `Q27C_*` (8개 카테고리) | AI 서비스 도움 정도 | 합산 → `AI_도움정도` (8~32) | 전원 응답. 높을수록 AI가 도움된다고 느낌 |
| `Q28_*` | AI 미이용 이유 (7개) | 이진변환 (값있음=1, NaN=0) | 이유 각각이 중요하므로 합산 안 함. 코드북 Q28_1~7 참고 |
| `Q31_*` | 정보화교육 수요 과정 (4개) | 이진변환 (값있음=1, NaN=0) | 어떤 과정을 원하는지가 중요하므로 합산 안 함 |
| `Q2K2_1`, `Q2K2_2` | 스마트패드/주변기기 보유 여부 | 이진변환 (1=보유, 나머지=0) | 보유=1, 미보유=0 |

---

## 5. 특수 처리 내역

| 항목 | 처리 내용 | 이유 |
|---|---|---|
| 일반국민/고령층 `ADQ5=2` 행 삭제 | 장애 있는 응답자 행 전체 제거 | 장애인 그룹과 중복. 장애인 파일에서 별도 수집됨 |
| 북한이탈주민/결혼이민자 제외 | 두 그룹 파일 전체 미사용 | 연령/성별 외 공통 변수가 너무 적어 다른 그룹과 비교 불가 |
| 농어민 `직업` 컬럼 값 = 6 고정 | BDQ1(농민/어민 구분)을 직업 코드 6번으로 고정 | 일반국민 ADQ3 코드 기준 6번 = 농림어업 종사자 |
| `Q28_*`, `Q31_*` 코드표 오류 | 코드표 레이블 무시하고 값 있음/없음으로만 이진변환 | 코드표가 전부 동일 레이블로 잘못 작성됨. 실제 값은 각 컬럼번호와 일치 |
| `ADQ8A_*` 코드표 오류 | 코드표 레이블 무시하고 이진변환 | 코드표가 전부 "배우자"로 잘못 작성됨 |
| 그룹별 공통 변수 매핑 | 그룹마다 다른 컬럼명을 연령/성별/직업/학력/가구구성형태/가구소득/거주지역으로 통일 | 5개 그룹 파일 합산 시 동일 정보를 같은 컬럼으로 관리하기 위함 |
| `join='inner'` 합산 | 5개 그룹 합칠 때 공통 컬럼만 유지 | 그룹 전용 컬럼(BDQ, CDQ 등)은 다른 그룹에 없어서 모델 feature로 쓸 수 없음 |
| `Q4C_1`, `Q4C_2` 중앙값 대체 | train 기준 중앙값 계산 후 val/test 동일 적용 | 진짜 결측(모름 응답자). 요금 부담 정도 변수로 디지털 접근성 지표에 활용 |

---
### 공통 변수 매핑 요약

| 공통 컬럼 | 일반국민/고령층 | 농어민 | 장애인 | 저소득층 |
|---|---|---|---|---|
| 연령 | ADQ1 | BDQ2 | CDQ1 | DDQ1 |
| 성별 | ADQ2 | BDQ3 | CDQ2 | DDQ2 |
| 직업 | ADQ3 | 6 고정 | CDQ6 | DDQ3 |
| 학력 | ADQ4 | BDQ4 | CDQ7 | DDQ4 |
| 가구구성형태 | ADQ8 | BDQ8 | CDQ13 | DDQ8 |
| 가구소득 | ADQ9 | BDQ9 | CDQ15 | DDQ9 |
| 거주지역 | ADQ101 | BDQ101 | CDQ161 | DDQ101 |
---
### 분할 결과

| 세트 | 행 수 | 컬럼 수 |
|---|---|---|
| train | 21,933 | 185 |
| val | 4,700 | 185 |
| test | 4,700 | 185 |
| **전체** | **31,333** | **185** |

### 그룹별 분포

| 그룹 | train | val | test |
|---|---|---|---|
| 일반국민 | 9,579 | 2,053 | 2,053 |
| 고령층 | 3,114 | 667 | 667 |
| 장애인 | 3,080 | 660 | 660 |
| 농어민 | 3,080 | 660 | 660 |
| 저소득층 | 3,080 | 660 | 660 |
| **합계** | **21,933** | **4,700** | **4,700** |
---
## 6. 결과

| 항목 | 내용 |
|---|---|
| 사용 연도 | 2024년, 2025년 |
| 사용 그룹 | 일반국민, 농어민, 장애인, 저소득층, 고령층 (5개) |
| 최종 행 수 | train + val + test 합산 |
| 분할 비율 | train 70% / val 15% / test 15% |
| stratify 기준 | GROUP (그룹 비율 유지) |
| 최종 결측값 | 0개 |
| 저장 파일 | `전처리 train_set.xlsx`,`전처리 test_set.xlsx`,`전처리 val_set.xlsx`,`train.csv`, `val.csv`, `test.csv` |



In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ─────────────────────────────────────────────
# 설정
# ─────────────────────────────────────────────

BASE_DIR   = Path.cwd().parent / "data/raw"
OUTPUT_DIR = Path.cwd().parent / "preprocessed"
OUTPUT_DIR.mkdir(exist_ok=True)

INPUT_FILES = {
    # 2025년
    "일반국민_2025": BASE_DIR / "2025" / "1) 일반국민_DATA.xlsx",
    "농어민_2025":   BASE_DIR / "2025" / "2) 농어민_DATA.xlsx",
    "장애인_2025":   BASE_DIR / "2025" / "3) 장애인_DATA.xlsx",
    "저소득층_2025": BASE_DIR / "2025" / "4) 저소득층_DATA.xlsx",
    "고령층_2025":   BASE_DIR / "2025" / "7) 고령층_DATA.xlsx",
    # 2024년
    "일반국민_2024": BASE_DIR / "2024" / "1) 일반국민_DATA.xlsx",
    "농어민_2024":   BASE_DIR / "2024" / "2) 농어민_DATA.xlsx",
    "장애인_2024":   BASE_DIR / "2024" / "3) 장애인_DATA.xlsx",
    "저소득층_2024": BASE_DIR / "2024" / "4) 저소득층_DATA.xlsx",
    "고령층_2024":   BASE_DIR / "2024" / "7) 고령층_DATA.xlsx",
}

RANDOM_STATE = 42
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15


print(f"BASE_DIR:   {BASE_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"BASE_DIR 존재 여부: {BASE_DIR.exists()}")

In [ ]:
'''전처리'''
# ─────────────────────────────────────────────
# Step 1: 공백/빈문자열 → NA 변환
# ─────────────────────────────────────────────

def blank_to_na(df):
    if df is None:
        raise ValueError("df가 None입니다. 먼저 데이터를 로드하세요.")
    new_df = df.replace(" ", pd.NA)
    print(f"  공백→NA 변환 후 결측값: {new_df.isnull().sum().sum()}")
    return new_df


# ─────────────────────────────────────────────
# Step 2: 불필요한 컬럼 제거
# ─────────────────────────────────────────────

def drop_useless_cols(df):
    original_cols = df.shape[1]

    et_cols       = df.columns[df.columns.str.endswith('ET')].tolist()
    cols_9997     = [c for c in df.columns if '9997' in str(c)]
    all_null_cols = [c for c in df.columns if df[c].isnull().all()]
    q4a_cols      = [c for c in df.columns if str(c).startswith('Q4A')]
    extra_drop = [c for c in ['응답ID', 'TYP', 'ADQ3A', 'ADQ8_1T'] if c in df.columns]
    drop_cols = et_cols + cols_9997 + all_null_cols + q4a_cols + extra_drop
    df = df.drop(columns=drop_cols)

    print(f'  컬럼 수: {original_cols} → {df.shape[1]}개 ({len(drop_cols)}개 제거)')
    return df


# ─────────────────────────────────────────────
# Step 3: 특수코드 처리
# ─────────────────────────────────────────────

def special_codes_to_na(df):
    """
    1. 일반 컬럼에서 9998/9999 값 → NA
    2. 컬럼명에 9998/9999 포함된 flag 컬럼 결측 → 0
    """
    special_flag_cols = [c for c in df.columns if '9998' in str(c) or '9999' in str(c)]
    numeric_cols      = df.select_dtypes(include=[np.number]).columns

    count = 0
    for col in numeric_cols:
        if col in special_flag_cols:
            continue
        mask   = df[col].isin([9998, 9999])
        count += mask.sum()
        df.loc[mask, col] = pd.NA

    df[special_flag_cols] = df[special_flag_cols].fillna(0)

    print(f'  특수코드→NA: {count}개 | flag 컬럼→0: {special_flag_cols}')
    return df


# ─────────────────────────────────────────────
# Step 4: 구조적 결측 → 0
# ─────────────────────────────────────────────

def fill_conditional_zero(df):
    """인터넷 비이용자 해당없음(Q12~Q19) → 0"""
    zero_fill_cols = [c for c in df.columns if
                      str(c).startswith(('Q12','Q13','Q14','Q15','Q16','Q17','Q18','Q19'))]
    df[zero_fill_cols] = df[zero_fill_cols].fillna(0).infer_objects(copy=False)
    print(f'  구조적 결측→0: {len(zero_fill_cols)}개 컬럼')
    return df



In [ ]:
# ─────────────────────────────────────────────
# Feature Engineering
# ─────────────────────────────────────────────

def feature_engineering(df):
    df = df.copy()
    # Q5K1: 혼자 해결 방법 - 이진변환
    for col in [c for c in df.columns if str(c).startswith('Q5K1')]:
        df[col] = df[col].notna().astype(int)

    # Q4B: 결합상품 가입 여부 - 이진변환
    for col in ['Q4B_1_1', 'Q4B_2_1']:
        if col in df.columns:
            df[col] = df[col].notna().astype(int)

    # ADQ8A: 가구구성형태 - 이진변환 (극소수 제거 후)
    df = df.drop(columns=[c for c in ['ADQ8A5','ADQ8A6','ADQ8A7','ADQ8A8'] if c in df.columns])
    for col in [c for c in df.columns if str(c).startswith('ADQ8A')]:
        df[col] = df[col].notna().astype(int)

    # Q27: 합산 후 원본 제거
    q27a_cols = [c for c in df.columns if 'Q27A' in str(c)]
    q27b_cols = [c for c in df.columns if 'Q27B' in str(c) and '9998' not in str(c)]
    q27c_cols = [c for c in df.columns if 'Q27C' in str(c) and '9998' not in str(c)]
    df['AI_인지']     = df[q27a_cols].notna().astype(int).sum(axis=1)
    df['AI_사용빈도'] = df[q27b_cols].fillna(0).sum(axis=1)
    df['AI_도움정도'] = df[q27c_cols].sum(axis=1)
    df = df.drop(columns=q27a_cols + q27b_cols + q27c_cols)

    # Q28: AI 비활용 이유 - 이진변환
    for col in [c for c in df.columns if str(c).startswith('Q28')]:
        df[col] = df[col].notna().astype(int)

    # Q31: 교육 수요 - 이진변환
    for col in [c for c in df.columns if str(c).startswith('Q31')]:
        df[col] = df[col].notna().astype(int)

    # Q2K1: 전체 제거
    df = df.drop(columns=[c for c in df.columns if str(c).startswith('Q2K1')])
    return df
   

# ─────────────────────────────────────────────
# 공통 변수 매핑
# ─────────────────────────────────────────────

COL_MAPPING = {
    "일반국민": {'ADQ1':'연령','ADQ2':'성별','ADQ3':'직업','ADQ4':'학력','ADQ8':'가구구성형태','ADQ9':'가구소득','ADQ101':'거주지역'},
    "농어민":   {'BDQ2':'연령','BDQ3':'성별','BDQ4':'학력','BDQ8':'가구구성형태','BDQ9':'가구소득','BDQ101':'거주지역'},
    "장애인":   {'CDQ1':'연령','CDQ2':'성별','CDQ6':'직업','CDQ7':'학력','CDQ13':'가구구성형태','CDQ15':'가구소득','CDQ161':'거주지역'},
    "저소득층": {'DDQ1':'연령','DDQ2':'성별','DDQ3':'직업','DDQ4':'학력','DDQ8':'가구구성형태','DDQ9':'가구소득','DDQ101':'거주지역'},
    "고령층":   {'ADQ1':'연령','ADQ2':'성별','ADQ3':'직업','ADQ4':'학력','ADQ8':'가구구성형태','ADQ9':'가구소득','ADQ101':'거주지역'},
}

def add_common_cols(df, group_label):
    for original_col, new_col in COL_MAPPING.get(group_label, {}).items():
        if original_col in df.columns:
            df[new_col] = df[original_col]

    if group_label == '농어민':
        df['직업'] = 6

    missing = [c for c in ['연령','성별','직업','학력','가구구성형태','가구소득','거주지역'] if c not in df.columns]
    if missing:
        print(f'누락된 공통 컬럼: {missing}')
    return df

In [ ]:
# ─────────────────────────────────────────────
# 그룹 전처리 파이프라인
# ─────────────────────────────────────────────

def preprocess_group(filepath, group_name):
    group_label, year = group_name.split('_')  # "일반국민_2025" → ("일반국민", "2025")

    df = pd.read_excel(filepath).copy()
    print(f'\n{"="*50}\n{group_name}\n{"="*50}')

    df = blank_to_na(df)

    # 일반국민, 고령층: 장애구분=2 행 제거
    if 'ADQ5' in df.columns:
        before = len(df)
        df = df[df['ADQ5'] != 2].reset_index(drop=True)
        print(f'  ADQ5=2 제거: {before - len(df)}명')

    df = drop_useless_cols(df)
    df = special_codes_to_na(df)
    df = fill_conditional_zero(df)
    df = feature_engineering(df)
    df = add_common_cols(df, group_label)

    # GROUP, YEAR 컬럼 맨 앞에 추가
    df = pd.concat([
        pd.DataFrame({'GROUP': group_label, 'YEAR': year}, index=df.index),
        df
    ], axis=1)

    print(f'  완료: {df.shape}')
    return df


In [ ]:
# ─────────────────────────────────────────────
# 실행
# ─────────────────────────────────────────────

all_groups = []
for group_name, filepath in INPUT_FILES.items():
    df_group = preprocess_group(filepath, group_name)
    all_groups.append(df_group)

df_all = pd.concat(all_groups, ignore_index=True, join='inner')
print(f'\n전체: {df_all.shape}')
print(df_all['GROUP'].value_counts())
print(df_all['YEAR'].value_counts())



일반국민_2025
  공백→NA 변환 후 결측값: 297290
  ADQ5=2 제거: 148명
  컬럼 수: 244 → 229개 (19개 제거)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\2175145177.py:90: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[special_flag_cols] = df[special_flag_cols].fillna(0)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\2175145177.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[zero_fill_cols] = df[zero_fill_cols].fillna(0).infer_objects(copy=False)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\301519971.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will

  특수코드→NA: 0개 | flag 컬럼→0: ['Q27C_9998']
  구조적 결측→0: 58개 컬럼
  완료: (6852, 210)

농어민_2025
  공백→NA 변환 후 결측값: 112851
  컬럼 수: 243 → 230개 (17개 제거)
  특수코드→NA: 0개 | flag 컬럼→0: ['Q27C_9998']


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\2175145177.py:90: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[special_flag_cols] = df[special_flag_cols].fillna(0)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\2175145177.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[zero_fill_cols] = df[zero_fill_cols].fillna(0).infer_objects(copy=False)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\301519971.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will

  구조적 결측→0: 58개 컬럼
  완료: (2200, 215)

장애인_2025
  공백→NA 변환 후 결측값: 116972
  컬럼 수: 253 → 240개 (18개 제거)


...

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\2175145177.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[zero_fill_cols] = df[zero_fill_cols].fillna(0).infer_objects(copy=False)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\301519971.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['AI_사용빈도'] = df[q27b_cols].fillna(0).sum(axis=1)



저소득층_2024
  공백→NA 변환 후 결측값: 101265
  컬럼 수: 239 → 227개 (14개 제거)
  특수코드→NA: 0개 | flag 컬럼→0: []
  구조적 결측→0: 57개 컬럼
  완료: (2200, 212)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\2175145177.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[zero_fill_cols] = df[zero_fill_cols].fillna(0).infer_objects(copy=False)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\301519971.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['AI_사용빈도'] = df[q27b_cols].fillna(0).sum(axis=1)



고령층_2024
  공백→NA 변환 후 결측값: 115264
  ADQ5=2 제거: 79명
  컬럼 수: 239 → 224개 (18개 제거)
  특수코드→NA: 0개 | flag 컬럼→0: []
  구조적 결측→0: 57개 컬럼
  완료: (2221, 207)

전체: (31333, 185)
GROUP
일반국민    13685
고령층      4448
농어민      4400
장애인      4400
저소득층     4400
Name: count, dtype: int64
YEAR
2025    15679
2024    15654
Name: count, dtype: int64


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\2175145177.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[zero_fill_cols] = df[zero_fill_cols].fillna(0).infer_objects(copy=False)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12540\301519971.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['AI_사용빈도'] = df[q27b_cols].fillna(0).sum(axis=1)


In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# train+val / test 분할
train_val, df_test = train_test_split(
    df_all,
    test_size=TEST_RATIO,
    stratify=df_all['GROUP'],
    random_state=RANDOM_STATE
)

# train / val 분할
val_size = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
df_train, df_val = train_test_split(
    train_val,
    test_size=val_size,
    stratify=train_val['GROUP'],
    random_state=RANDOM_STATE
)

print(f'\ntrain: {df_train.shape} | GROUP 분포:')
print(df_train['GROUP'].value_counts())
print(f'\nval:   {df_val.shape} | GROUP 분포:')
print(df_val['GROUP'].value_counts())
print(f'\ntest:  {df_test.shape} | GROUP 분포:')
print(df_test['GROUP'].value_counts())





train: (21933, 185) | GROUP 분포:
GROUP
일반국민    9579
고령층     3114
장애인     3080
농어민     3080
저소득층    3080
Name: count, dtype: int64

val:   (4700, 185) | GROUP 분포:
GROUP
일반국민    2053
고령층      667
저소득층     660
장애인      660
농어민      660
Name: count, dtype: int64

test:  (4700, 185) | GROUP 분포:
GROUP
일반국민    2053
고령층      667
저소득층     660
장애인      660
농어민      660
Name: count, dtype: int64


In [ ]:
from sklearn.impute import SimpleImputer

# ── 1. 수치형 변환 ─────────────────────────────
# object/string으로 남아있는 컬럼들 중 숫자형으로 바꿀 수 있는 값 변환
# 전처리 후 분할된 세 세트 모두 변환

def convert_to_numeric(df):
    obj_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in obj_cols:
        # GROUP 같은 범주 문자열 컬럼은 숫자 변환 대상에서 제외
        if col == 'GROUP':
            continue
        df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f'  수치형 변환: {len(obj_cols)}개 컬럼')
    return df

df_train = convert_to_numeric(df_train)
df_val   = convert_to_numeric(df_val)
df_test  = convert_to_numeric(df_test)


# ── 2. Q4C, Q11 결측값 중앙값 대체 ───────────────
# train 기준으로만 계산 후 세 세트 모두 적용

target_cols = (
    [c for c in df_train.columns if str(c).startswith('Q4C')] +
    [c for c in df_train.columns if str(c).startswith('Q11')]
)
target_cols = [c for c in target_cols if df_train[c].isnull().sum() > 0]

if target_cols:
    imputer = SimpleImputer(strategy='median')
    imputer.fit(df_train[target_cols])
    df_train[target_cols] = imputer.transform(df_train[target_cols])
    df_val[target_cols]   = imputer.transform(df_val[target_cols])
    df_test[target_cols]  = imputer.transform(df_test[target_cols])
    print(f'  중앙값 대체 완료: {target_cols}')
else:
    print('  결측값 없음')


# ── 확인 ───────────────────────────────────────
print(f'\ntrain 잔여 결측: {df_train.isnull().sum().sum()}')
print(f'val 잔여 결측:   {df_val.isnull().sum().sum()}')
print(f'test 잔여 결측:  {df_test.isnull().sum().sum()}')

# 저장

# df_train.to_csv(OUTPUT_DIR / 'train.csv', index=False, encoding='utf-8-sig')
# df_val.to_csv(  OUTPUT_DIR / 'val.csv',   index=False, encoding='utf-8-sig')
# df_test.to_csv( OUTPUT_DIR / 'test.csv',  index=False, encoding='utf-8-sig')
# print('\n저장 완료')

  수치형 변환: 7개 컬럼
  수치형 변환: 7개 컬럼
  수치형 변환: 7개 컬럼
  중앙값 대체 완료: ['Q4C_1', 'Q4C_2', 'Q11_1', 'Q11_2', 'Q11_3']

train 잔여 결측: 0
val 잔여 결측:   0
test 잔여 결측:  0

저장 완료


In [ ]:
# # 전처리 완료 저장
# df_train.to_excel(OUTPUT_DIR / '전처리 train set.xlsx', index=False)
# df_val.to_excel(OUTPUT_DIR / '전처리 val set.xlsx', index=False)
# df_test.to_excel(OUTPUT_DIR / '전처리 test set.xlsx', index=False)
# print(f'\n저장 완료: {OUTPUT_DIR / "전처리set.xlsx"}')


저장 완료: preprocessed\전처리set.xlsx
